# Review segments and assigned topics

Run the cells below, then choose a **document index** or title. Switch between
**Both**, **Semantic**, and **Fixed**. Both shows the same talk side by side with each
model's native boundaries. Cards display full text, topic IDs and Estonian names,
timestamps, audio links and cluster membership strength.

Use **Unassigned only**, **Weak / unassigned**, or the search field to focus your review.
This notebook reads saved results; it does not fit models or call an API. Document
indices start at **0** and are sorted by title, with episode ID as a tie-breaker.

When saved Jev predictions are present, semantic cards also show the detailed primary topic,
choice probability, confidence and top three alternatives beside the original
cluster assignment. Unsampled segments are marked. Jev scores have not been
validated for these Estonian transcripts; they are separate from cluster strength.

Choose **Original vs shorter** to compare the three-talk shorter semantic pilot.
This view shows all segments and ignores the original-mode filters. Short segments
have Jev predictions; their original labels are shown only as temporal overlaps.

Choose **Experiments**, then select any two variants on the left and right.
Set **From min / To min** to compare a passage; To min = 0 shows the rest of
the recording. Documents **0, 1, 3** are the pilot talks. Sentence cuts use
punctuation from the transcript, not guaranteed story or speaker boundaries.

Independent topic-model refits are available for each experiment at seeds **42**
and **7**. Select **Topic model**, **Jev**, or both. Refit labels are fresh keyword
labels; topic numbers are not aligned across variants. The model summary reports
topic counts, unassigned time, and topic changes (including transitions to/from -1).

The experiment view also shows all five refitted topic timelines on one recording
axis. Hover for keywords and membership strength. Gray means unassigned; colors
and numeric IDs are local to each fit. Use the time window to zoom the timeline.

When the full Jev export is available, Original view shows detailed Jev labels for all current semantic segments and their dictionary-based high-level categories. Fine model memberships remain unchanged.


In [ ]:
import os
import hashlib
import json
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, clear_output, display

from arvamusfestivali_transcripts.topic_analysis.segment_review import (
    document_catalog,
    segment_review_html,
    semantic_length_review_html,
    segmentation_experiment_review_html,
)

# Override TOPIC_REVIEW_DATASET to inspect another saved run.
project_root = Path(os.environ.get("TOPIC_ANALYSIS_PROJECT_ROOT", Path.cwd())).resolve()
if not (project_root / "data" / "topic-analysis").exists():
    if (project_root.parent / "data" / "topic-analysis").exists():
        project_root = project_root.parent
DATASET_PATH = Path(os.environ.get(
    "TOPIC_REVIEW_DATASET",
    str(project_root / "data/topic-analysis/results/segment-dataset/all-segments.parquet"),
))
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Saved segment dataset missing: {DATASET_PATH}")
segments = pd.read_parquet(DATASET_PATH)
# Load the optional Jev sidecar; the review notebook makes no API calls.
JEV_ALL_PATH = project_root / "data/topic-analysis/results/jev-semantic-all/assignments.parquet"
JEV_PATH = JEV_ALL_PATH if JEV_ALL_PATH.exists() else project_root / "data/topic-analysis/results/jev-semantic-sample/assignments.parquet"
if JEV_PATH.exists() and "jev_topic_id" not in segments.columns:
    manifest = json.loads(JEV_PATH.with_name("manifest.json").read_text())
    if hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest() != manifest["source_sha256"]:
        raise ValueError("Jev sidecar belongs to a different fitted dataset; choose its source dataset.")
    jev = pd.read_parquet(JEV_PATH)
    segments = segments.merge(jev, on="segment_key", how="left", validate="one_to_one")
    print(f"Loaded {segments.jev_topic_id.notna().sum():,} Jev predictions.")

segments["episode_id"] = segments.episode_id.astype(str)
documents = document_catalog(segments)
if documents.empty:
    raise ValueError("The dataset contains no documents")
print(f"Loaded {len(segments):,} segments from {len(documents)} documents.")
print(DATASET_PATH)
# Pilot document indices are shown here; full-corpus indices appear in the catalog cell.
if "jev_topic_id" in segments.columns and not JEV_ALL_PATH.exists():
    sampled_ids = set(segments.loc[segments.jev_topic_id.notna(), "episode_id"])
    display(documents[documents.episode_id.isin(sampled_ids)].rename_axis("document_index"))

SHORT_PATH = project_root / "data/topic-analysis/results/jev-short-semantic-sample/review-segments.parquet"
short_segments = pd.read_parquet(SHORT_PATH) if SHORT_PATH.exists() else None
if short_segments is not None:
    short_segments["episode_id"] = short_segments.episode_id.astype(str)
    print(f"Shorter semantic pilot: {len(short_segments)} segments. Select Original vs shorter below.")

# Jev assignments use the original vocabulary; independent refits are loaded below.
experiment_variants = {"original": segments}
if short_segments is not None:
    experiment_variants["short3"] = short_segments
EXPERIMENT_ROOT = project_root / "data/topic-analysis/results/segmentation-experiments"
for variant in ("cue5", "sentence3", "sentence5"):
    path = EXPERIMENT_ROOT / variant / "review-segments.parquet"
    if path.exists():
        manifest = json.loads(path.with_name("segmentation-manifest.json").read_text())
        if hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest() != manifest["original_dataset_sha256"]:
            raise ValueError("Segmentation experiments belong to another original fitted dataset.")
        experiment_variants[variant] = pd.read_parquet(path)
SUMMARY_PATH = EXPERIMENT_ROOT / "comparison-summary.csv"
if SUMMARY_PATH.exists():
    print("Experiment summary: probabilities and abstention are descriptive, not accuracy.")
    display(pd.read_csv(SUMMARY_PATH))

# Independent topic-model refits, retaining their own run-scoped topic IDs.
TOPIC_MODEL_ROOT = EXPERIMENT_ROOT / "topic-models"
refit_variants = {}
for fit_seed in (42, 7):
    for variant in ("original", "short3", "cue5", "sentence3", "sentence5"):
        path = TOPIC_MODEL_ROOT / variant / f"seed-{fit_seed}" / "review-segments.parquet"
        if path.exists():
            fitted = pd.read_parquet(path)
            fitted["episode_id"] = fitted.episode_id.astype(str)
            refit_variants[(variant, fit_seed)] = fitted
MODEL_SUMMARY = TOPIC_MODEL_ROOT / "comparison-summary.csv"
if MODEL_SUMMARY.exists():
    print("Independent three-talk topic-model fits: topic IDs and keyword labels are local to each fit.")
    display(pd.read_csv(MODEL_SUMMARY)[["variant", "seed", "segments", "topic_count",
        "outlier_time_pct", "topic_changes", "mean_changes_per_hour"]])
    display(pd.read_csv(TOPIC_MODEL_ROOT / "seed-stability.csv"))

from arvamusfestivali_transcripts.topic_analysis.experiment_dynamics import plot_variant_topic_timelines


In [ ]:
# Catalog: these indices select the same recording in both modes.
display(documents.rename_axis("document_index"))

In [ ]:
document_index = widgets.BoundedIntText(
    value=0, min=0, max=len(documents) - 1, description="Doc index:",
)
document_title = widgets.Dropdown(
    options=[(f"{i}: {row.talk_name}", i) for i, row in documents.iterrows()],
    value=0, description="Document:", layout=widgets.Layout(width="95%"),
)
index_link = widgets.link((document_index, "value"), (document_title, "value"))
previous = widgets.Button(description="← Previous")
following = widgets.Button(description="Next →")
previous.on_click(lambda _: setattr(document_index, "value", max(0, document_index.value - 1)))
following.on_click(lambda _: setattr(
    document_index, "value", min(len(documents) - 1, document_index.value + 1)
))
mode = widgets.ToggleButtons(
    options=[("Both", "both"), ("Semantic", "semantic"), ("Fixed", "fixed")],
    value="semantic" if JEV_ALL_PATH.exists() else "both", description="Mode:",
)
unassigned = widgets.Checkbox(value=False, description="Unassigned only")
weak = widgets.Checkbox(value=False, description="Weak / unassigned")
threshold = widgets.FloatSlider(
    value=0.5, min=0, max=1, step=0.05, description="Weak ≤", continuous_update=False,
)
search = widgets.Text(
    placeholder="Search segment text or topic names", description="Search:",
    continuous_update=False, layout=widgets.Layout(width="95%"),
)
comparison = widgets.ToggleButtons(
    options=[("Original", "original"), ("Original vs shorter", "shorter"), ("Experiments", "experiments")],
    value="original" if JEV_ALL_PATH.exists() else ("experiments" if "sentence5" in experiment_variants else "original"), description="View:",
)
variant_labels = {
    "original": "Original semantic", "short3": "Cue cuts · 3 minutes",
    "cue5": "Cue cuts · 5 minutes", "sentence3": "Sentence cuts · 3 minutes",
    "sentence5": "Sentence cuts · 5 minutes",
}
variant_options = [(variant_labels[key], key) for key in experiment_variants]
experiment_left = widgets.Dropdown(options=variant_options,
    value="cue5" if "cue5" in experiment_variants else "original", description="Left:")
experiment_right = widgets.Dropdown(options=variant_options,
    value="sentence5" if "sentence5" in experiment_variants else "original", description="Right:")
window_start = widgets.FloatText(value=0, description="From min:")
window_end = widgets.FloatText(value=0, description="To min:")  # 0 means recording end
fit_seed = widgets.Dropdown(options=[42, 7], value=42, description="Fit seed:")
assignment_source = widgets.ToggleButtons(
    options=[("Topic model + Jev", "both"), ("Topic model", "topic_model"), ("Jev", "jev")],
    value="both", description="Labels:",
)
review_output = widgets.Output()


def show_review(change=None):
    with review_output:
        clear_output(wait=True)
        if comparison.value == "experiments":
            active_variants = {
                key: refit_variants.get((key, fit_seed.value), value)
                for key, value in experiment_variants.items()
            }
            if any("cluster_topic_id" in data.columns for data in active_variants.values()):
                episode = documents.iloc[document_index.value].episode_id
                timeline = plot_variant_topic_timelines(active_variants, episode,
                    title=f"Native topic timelines · seed {fit_seed.value}")
                if window_end.value > window_start.value:
                    timeline.update_xaxes(range=[window_start.value, window_end.value])
                display(timeline)
            display(HTML(segmentation_experiment_review_html(
                segments, active_variants, document_index.value,
                left=experiment_left.value, right=experiment_right.value,
                assignment_source=assignment_source.value,
                start_seconds=window_start.value * 60,
                end_seconds=window_end.value * 60 if window_end.value > 0 else None,
            )))
            return
        if comparison.value == "shorter":
            if short_segments is None:
                print("Run the shorter semantic pilot first.")
            else:
                display(HTML(semantic_length_review_html(segments, short_segments, document_index.value)))
            return
        display(HTML(segment_review_html(
            segments, document_index.value, mode=mode.value,
            only_unassigned=unassigned.value, weak_only=weak.value,
            weak_threshold=threshold.value, search=search.value,
        )))


for control in (document_index, mode, unassigned, weak, threshold, search, comparison, experiment_left, experiment_right, window_start, window_end, fit_seed, assignment_source):
    control.observe(show_review, names="value")
display(widgets.VBox([
    document_title,
    widgets.HBox([previous, document_index, following]),
    comparison,
    widgets.HBox([experiment_left, experiment_right]),
    widgets.HBox([window_start, window_end]),
    widgets.HBox([fit_seed, assignment_source]),
    mode,
    widgets.HBox([unassigned, weak]),
    threshold,
    search,
    review_output,
]))
show_review()

## Direct selection without widgets

The cell below also works in notebook viewers without interactive widget support.
Change `DOC_INDEX` and `MODE`, then rerun it. This view has independent settings.

In [ ]:
DOC_INDEX = 0
MODE = "both"  # "both", "semantic", or "fixed"
# Uncomment to display a second, non-widget review:
# display(HTML(segment_review_html(segments, DOC_INDEX, mode=MODE)))
# Direct shorter comparison, without widgets:
# display(HTML(semantic_length_review_html(segments, short_segments, DOC_INDEX)))

# Direct experiment comparison:
# display(HTML(segmentation_experiment_review_html(segments, experiment_variants, DOC_INDEX,
#     left="cue5", right="sentence5", start_seconds=75*60, end_seconds=85*60)))


## Broader 18-talk pilot

Compare original segmentation with sentence-aware cuts around five minutes. These independent fits use seeds 42, 7 and 19 and fresh keyword labels. The sample covers varied subjects and recording formats; it is not a representative accuracy estimate. Topic IDs/colors differ across fits. Review coherent topics and meaningful changes alongside unassigned time and seed stability. Scores in manual-review.csv remain blank for your review.

In [ ]:
BROAD_ROOT = EXPERIMENT_ROOT / "broader-pilot"
broad_original = pd.read_parquet(BROAD_ROOT / "original.parquet")
broad_original["episode_id"] = broad_original.episode_id.astype(str)
broad_documents = document_catalog(broad_original)
broad_fits = {}
for seed in (42, 7, 19):
    for variant in ("original", "sentence5"):
        frame = pd.read_parquet(BROAD_ROOT / "topic-models" / variant / f"seed-{seed}" / "review-segments.parquet")
        frame["episode_id"] = frame.episode_id.astype(str)
        broad_fits[(variant, seed)] = frame
display(pd.read_csv(BROAD_ROOT / "sample-talks.csv"))
if (BROAD_ROOT / "qualitative-examples.csv").exists():
    print("Six qualitative Codex readings; not human accuracy scores:")
    display(pd.read_csv(BROAD_ROOT / "qualitative-examples.csv"))
display(pd.read_csv(BROAD_ROOT / "topic-models/comparison-summary.csv"))
display(pd.read_csv(BROAD_ROOT / "topic-models/seed-stability.csv"))
broad_doc = widgets.Dropdown(options=[(row.talk_name, i) for i, row in broad_documents.iterrows()], description="Talk:", layout=widgets.Layout(width="95%"))
broad_seed = widgets.Dropdown(options=[42, 7, 19], description="Seed:")
broad_start = widgets.FloatText(value=10, description="From min:")
broad_end = widgets.FloatText(value=18, description="To min:")
broad_output = widgets.Output()
def show_broad(change=None):
    with broad_output:
        clear_output(wait=True)
        fits = {variant: broad_fits[(variant, broad_seed.value)] for variant in ("original", "sentence5")}
        episode = broad_documents.iloc[broad_doc.value].episode_id
        display(plot_variant_topic_timelines(fits, episode, title=f"18-talk pilot · seed {broad_seed.value}"))
        display(HTML(segmentation_experiment_review_html(broad_original, fits, broad_doc.value,
            left="original", right="sentence5", assignment_source="topic_model",
            start_seconds=broad_start.value*60, end_seconds=broad_end.value*60 if broad_end.value>0 else None)))
for control in (broad_doc, broad_seed, broad_start, broad_end):
    control.observe(show_broad, names="value")
display(widgets.VBox([broad_doc, widgets.HBox([broad_seed, broad_start, broad_end]), broad_output]))
show_broad()


## High-level categories for the current segmentation

The 236 current semantic topics are grouped into 20 reviewable parent categories. Fine IDs and transcript boundaries stay unchanged. The dictionary applies to this exact fitted vocabulary. `-1` maps to `unassigned`. Jev results below illustrate the lookup. Full-corpus labels load automatically when available.

In [ ]:
HIGH_LEVEL_ROOT = project_root / "data/topic-analysis/results/high-level-topics"
high_level_manifest = json.loads((HIGH_LEVEL_ROOT / "manifest.json").read_text())
if hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest() != high_level_manifest["source_dataset_sha256"]:
    raise ValueError("High-level mapping belongs to another source dataset.")
fine_to_high_level = json.loads((HIGH_LEVEL_ROOT / "topic-to-high-level.json").read_text())
topic_key_to_high_level = json.loads((HIGH_LEVEL_ROOT / "topic-key-to-high-level.json").read_text())
high_level_categories = pd.read_csv(HIGH_LEVEL_ROOT / "categories.csv")
display(high_level_categories)
high_level_topic_mapping = pd.read_csv(HIGH_LEVEL_ROOT / "topic-mapping.csv")
display(high_level_topic_mapping[["topic_id", "topic_name_et", "high_level_id", "high_level_name_et"]])
if "jev_topic_id" in segments:
    jev_parent_preview = segments[segments.jev_topic_id.notna()].copy()
    jev_parent_preview["jev_high_level_id"] = jev_parent_preview.jev_topic_id.map(
        lambda topic: fine_to_high_level[str(int(topic))])
    display(jev_parent_preview[["talk_name", "start_seconds", "jev_topic_name", "jev_high_level_id"]].head(20))
print("Complete mapping: 236 fine topics → 20 categories; -1 → unassigned.")


## Jev treemaps: segments and distinct talks

High-level categories contain the detailed topics assigned directly by Jev. Click a category to drill down; use the path bar to return. Gray is unassigned.

For talks, each topic and category displays its distinct recording count. Talks can contain several topics, so category areas sum child topic–talk memberships while category labels count the union. Hover shows segment counts, distinct talks and area weights.

In [ ]:
import plotly.io as pio
TREEMAP_ROOT = project_root / "data/topic-analysis/results/jev-semantic-all/treemaps"
treemap_manifest = json.loads((TREEMAP_ROOT / "manifest.json").read_text())
treemap_source = TREEMAP_ROOT.parent / "review-segments.parquet"
if hashlib.sha256(treemap_source.read_bytes()).hexdigest() != treemap_manifest["source_sha256"]:
    raise ValueError("Treemaps belong to another Jev export.")
treemap_metric = widgets.ToggleButtons(options=[("Segment counts", "segments"), ("Distinct talks", "talks")], description="Treemap:")
treemap_output = widgets.Output()
def show_jev_treemap(change=None):
    with treemap_output:
        clear_output(wait=True)
        display(pio.from_json((TREEMAP_ROOT / f"{treemap_metric.value}.plotly.json").read_text()))
treemap_metric.observe(show_jev_treemap, names="value")
display(widgets.VBox([treemap_metric, treemap_output]))
show_jev_treemap()


## Jev topic bubble maps

Each bubble is a detailed Jev topic; colors identify dictionary high-level categories. Switch between segment count and distinct talk count. Bubble **area** follows the count. Both views use exactly the same 2D positions, derived from normalized means of the genuine BGE embeddings of Jev-assigned segments. UMAP distances are approximate; unassigned content is excluded. Hover for names and both counts; click the legend to filter categories.

In [ ]:
import plotly.io as pio
BUBBLE_ROOT = project_root / "data/topic-analysis/results/jev-semantic-all/bubble-maps"
bubble_manifest = json.loads((BUBBLE_ROOT / "manifest.json").read_text())
if hashlib.sha256((BUBBLE_ROOT.parent / "review-segments.parquet").read_bytes()).hexdigest() != bubble_manifest["source_sha256"]:
    raise ValueError("Bubble map belongs to another Jev export.")
bubble_metric = widgets.ToggleButtons(options=[("Size by segments", "segments"), ("Size by distinct talks", "talks")], description="Bubbles:")
bubble_output = widgets.Output()
def show_jev_bubbles(change=None):
    with bubble_output:
        clear_output(wait=True)
        display(pio.from_json((BUBBLE_ROOT / f"{bubble_metric.value}.plotly.json").read_text()))
bubble_metric.observe(show_jev_bubbles, names="value")
display(widgets.VBox([bubble_metric, bubble_output]))
show_jev_bubbles()
